In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    "../data/processed/superstore_clean.csv",
    parse_dates=["order_date", "ship_date", "month_start"]
)

print("Rows:", len(df))
print("Columns:", len(df.columns))

Rows: 9994
Columns: 29


In [2]:
df["profit_margin"] = (
    df["profit"] / df["revenue"] * 100
)

In [3]:
def classify_profitability(margin):
    if margin < 0:
        return "Loss"
    elif margin < 5:
        return "Critical"
    elif margin < 10:
        return "Low"
    elif margin < 20:
        return "Healthy"
    else:
        return "High"


df["profitability_status"] = df["profit_margin"].apply(
    classify_profitability
)

In [4]:
profitability_risk = (
    df.groupby("profitability_status")
      .agg(
          transactions=("row_id", "count"),
          revenue=("revenue", "sum"),
          profit=("profit", "sum"),
          units=("quantity", "sum")
      )
      .reset_index()
)

In [5]:
profitability_risk["revenue_share"] = (
    profitability_risk["revenue"]
    / profitability_risk["revenue"].sum()
    * 100
)

profitability_risk["transaction_share"] = (
    profitability_risk["transactions"]
    / profitability_risk["transactions"].sum()
    * 100
)

profitability_risk["profit_margin"] = (
    profitability_risk["profit"]
    / profitability_risk["revenue"]
    * 100
)

In [6]:
profitability_risk

,profitability_status,transactions,revenue,profit,units,revenue_share,transaction_share,profit_margin
0,Critical,350,1.310900e+05,2921.0734,1301,5.706509,3.502101,2.228297
1,Healthy,1097,3.674590e+05,49304.9523,4198,15.995947,10.976586,13.417809
2,High,5963,1.087888e+06,373016.0363,22691,47.357112,59.665799,34.288092
3,Loss,1871,4.687072e+05,-156131.2857,7040,20.403403,18.721233,-33.311052
4,Low,713,2.420567e+05,17286.2454,2643,10.537029,7.134281,7.141403


In [7]:
status_order = [
    "Loss",
    "Critical",
    "Low",
    "Healthy",
    "High"
]

profitability_risk["profitability_status"] = pd.Categorical(
    profitability_risk["profitability_status"],
    categories=status_order,
    ordered=True
)

profitability_risk = profitability_risk.sort_values(
    "profitability_status"
)

profitability_risk

,profitability_status,transactions,revenue,profit,units,revenue_share,transaction_share,profit_margin
3,Loss,1871,4.687072e+05,-156131.2857,7040,20.403403,18.721233,-33.311052
0,Critical,350,1.310900e+05,2921.0734,1301,5.706509,3.502101,2.228297
4,Low,713,2.420567e+05,17286.2454,2643,10.537029,7.134281,7.141403
1,Healthy,1097,3.674590e+05,49304.9523,4198,15.995947,10.976586,13.417809
2,High,5963,1.087888e+06,373016.0363,22691,47.357112,59.665799,34.288092


In [8]:
loss_transactions = df[df["profitability_status"] == "Loss"]

print(
    "Loss-making transactions:",
    len(loss_transactions)
)

print(
    "Loss transaction rate:",
    len(loss_transactions) / len(df) * 100,
    "%"
)

Loss-making transactions: 1871
Loss transaction rate: 18.721232739643785 %


In [9]:
critical_transactions = df[
    df["profitability_status"].isin(
        ["Loss", "Critical"]
    )
]

print(
    "Loss + Critical transactions:",
    len(critical_transactions)
)

print(
    "Loss + Critical rate:",
    len(critical_transactions) / len(df) * 100,
    "%"
)

Loss + Critical transactions: 2221
Loss + Critical rate: 22.223334000400243 %


In [10]:
profitability_category_risk = (
    critical_transactions
    .groupby("category")
    .agg(
        transactions=("row_id", "count"),
        revenue=("revenue", "sum"),
        profit=("profit", "sum"),
        avg_discount=("discount", "mean")
    )
    .reset_index()
)

profitability_category_risk["profit_margin"] = (
    profitability_category_risk["profit"]
    / profitability_category_risk["revenue"]
    * 100
)

profitability_category_risk

,category,transactions,revenue,profit,avg_discount,profit_margin
0,Furniture,817,314932.9288,-59807.4897,0.345704,-18.990548
1,Office Supplies,1081,158926.1890,-55015.4301,0.507308,-34.616969
2,Technology,323,125938.0040,-38387.2925,0.295356,-30.481103


In [11]:
profitability_region_risk = (
    critical_transactions
    .groupby("region")
    .agg(
        transactions=("row_id", "count"),
        revenue=("revenue", "sum"),
        profit=("profit", "sum"),
        avg_discount=("discount", "mean")
    )
    .reset_index()
)

profitability_region_risk["profit_margin"] = (
    profitability_region_risk["profit"]
    / profitability_region_risk["revenue"]
    * 100
)

profitability_region_risk

,region,transactions,revenue,profit,avg_discount,profit_margin
0,Central,789,158842.4148,-55969.2294,0.521470,-35.235695
1,East,652,187676.0420,-49076.9582,0.391564,-26.149826
2,South,312,111630.4190,-27156.9199,0.418429,-24.327527
3,West,468,141648.2460,-21007.1048,0.275534,-14.830473


In [12]:
profitability_segment_risk = (
    critical_transactions
    .groupby("segment")
    .agg(
        transactions=("row_id", "count"),
        revenue=("revenue", "sum"),
        profit=("profit", "sum"),
        avg_discount=("discount", "mean")
    )
    .reset_index()
)

profitability_segment_risk["profit_margin"] = (
    profitability_segment_risk["profit"]
    / profitability_segment_risk["revenue"]
    * 100
)

profitability_segment_risk

,segment,transactions,revenue,profit,avg_discount,profit_margin
0,Consumer,1190,319383.7995,-83276.8073,0.413538,-26.074211
1,Corporate,658,175918.3068,-43820.8984,0.425000,-24.909800
2,Home Office,373,104495.0155,-26112.5066,0.414155,-24.989237


In [13]:
def classify_discount(discount):
    if discount == 0:
        return "0%"
    elif discount <= 0.10:
        return "1-10%"
    elif discount <= 0.20:
        return "11-20%"
    elif discount <= 0.30:
        return "21-30%"
    elif discount <= 0.50:
        return "31-50%"
    else:
        return "51%+"

df["discount_band"] = df["discount"].apply(classify_discount)

In [14]:
df["discount_band"].value_counts()

discount_band
0%        4798
11-20%    3709
51%+       856
31-50%     310
21-30%     227
1-10%       94
Name: count, dtype: int64

In [15]:
discount_risk = (
    df.groupby("discount_band")
      .agg(
          transactions=("row_id", "count"),
          revenue=("revenue", "sum"),
          profit=("profit", "sum"),
          units=("quantity", "sum")
      )
      .reset_index()
)

In [16]:
discount_risk["profit_margin"] = (
    discount_risk["profit"]
    / discount_risk["revenue"]
    * 100
)

In [17]:
discount_risk["transaction_share"] = (
    discount_risk["transactions"]
    / discount_risk["transactions"].sum()
    * 100
)

In [18]:
discount_risk["revenue_share"] = (
    discount_risk["revenue"]
    / discount_risk["revenue"].sum()
    * 100
)

In [19]:
discount_risk

,discount_band,transactions,revenue,profit,units,profit_margin,transaction_share,revenue_share
0,0%,4798,1.087908e+06,320987.6032,18267,29.505019,48.008805,47.358004
1,1-10%,94,5.436935e+04,9029.1770,373,16.607108,0.940564,2.366765
2,11-20%,3709,7.921529e+05,91756.2975,13858,11.583155,37.112267,34.483397
3,21-30%,227,1.032267e+05,-10369.2774,849,-10.045155,2.271363,4.493584
4,31-50%,310,1.953148e+05,-48447.7273,1177,-24.804950,3.101861,8.502293
5,51%+,856,6.422874e+04,-76559.0513,3349,-119.197502,8.565139,2.795957


In [20]:
discount_order = [
    "0%",
    "1-10%",
    "11-20%",
    "21-30%",
    "31-50%",
    "51%+"
]

discount_risk["discount_band"] = pd.Categorical(
    discount_risk["discount_band"],
    categories=discount_order,
    ordered=True
)

discount_risk = discount_risk.sort_values(
    "discount_band"
)

discount_risk

,discount_band,transactions,revenue,profit,units,profit_margin,transaction_share,revenue_share
0,0%,4798,1.087908e+06,320987.6032,18267,29.505019,48.008805,47.358004
1,1-10%,94,5.436935e+04,9029.1770,373,16.607108,0.940564,2.366765
2,11-20%,3709,7.921529e+05,91756.2975,13858,11.583155,37.112267,34.483397
3,21-30%,227,1.032267e+05,-10369.2774,849,-10.045155,2.271363,4.493584
4,31-50%,310,1.953148e+05,-48447.7273,1177,-24.804950,3.101861,8.502293
5,51%+,856,6.422874e+04,-76559.0513,3349,-119.197502,8.565139,2.795957


In [21]:
for _, row in discount_risk.iterrows():
    print(
        f"{row['discount_band']}: "
        f"Margin = {row['profit_margin']:.2f}% | "
        f"Profit = ${row['profit']:,.2f}"
    )

0%: Margin = 29.51% | Profit = $320,987.60
1-10%: Margin = 16.61% | Profit = $9,029.18
11-20%: Margin = 11.58% | Profit = $91,756.30
21-30%: Margin = -10.05% | Profit = $-10,369.28
31-50%: Margin = -24.80% | Profit = $-48,447.73
51%+: Margin = -119.20% | Profit = $-76,559.05


In [22]:
high_discount = df[
    df["discount"] > 0.20
].copy()

In [23]:
print("High-discount transactions:", len(high_discount))

print(
    "High-discount transaction rate:",
    len(high_discount) / len(df) * 100,
    "%"
)

High-discount transactions: 1393
High-discount transaction rate: 13.938363017810687 %


In [24]:
print(
    "High-discount revenue:",
    high_discount["revenue"].sum()
)

print(
    "High-discount profit:",
    high_discount["profit"].sum()
)

High-discount revenue: 362770.1498
High-discount profit: -135376.056


In [25]:
high_discount_loss = high_discount[
    high_discount["profit"] < 0
].copy()

In [26]:
print(
    "High-discount loss transactions:",
    len(high_discount_loss)
)

print(
    "High-discount loss:",
    high_discount_loss["profit"].sum()
)

print(
    "Revenue associated with high-discount losses:",
    high_discount_loss["revenue"].sum()
)

High-discount loss transactions: 1348
High-discount loss: -138515.2429
Revenue associated with high-discount losses: 330616.42480000004


In [31]:
discount_category_risk = (
    high_discount
    .assign(is_loss=high_discount["profit"] < 0)
    .groupby("category")
    .agg(
        transactions=("row_id", "count"),
        revenue=("revenue", "sum"),
        profit=("profit", "sum"),
        avg_discount=("discount", "mean"),
        loss_transactions=("is_loss", "sum")
    )
    .reset_index()
)

discount_category_risk["profit_margin"] = (
    discount_category_risk["profit"]
    / discount_category_risk["revenue"]
    * 100
)

discount_category_risk

,category,transactions,revenue,profit,avg_discount,loss_transactions,profit_margin
0,Furniture,542,195150.7408,-54477.7561,0.425258,527,-27.915731
1,Office Supplies,680,39523.1470,-47140.1376,0.744118,680,-119.272227
2,Technology,171,128096.2620,-33758.1623,0.444444,141,-26.353745


In [32]:
discount_region_risk = (
    high_discount
    .assign(is_loss=high_discount["profit"] < 0)
    .groupby("region")
    .agg(
        transactions=("row_id", "count"),
        revenue=("revenue", "sum"),
        profit=("profit", "sum"),
        avg_discount=("discount", "mean"),
        loss_transactions=("is_loss", "sum")
    )
    .reset_index()
)

In [33]:
discount_region_risk["profit_margin"] = (
    discount_region_risk["profit"]
    / discount_region_risk["revenue"]
    * 100
)

discount_region_risk

,region,transactions,revenue,profit,avg_discount,loss_transactions,profit_margin
0,Central,643,129134.0988,-52392.2744,0.606128,626,-40.571991
1,East,461,145870.3970,-42088.6076,0.507592,433,-28.853426
2,South,171,70716.9930,-24880.4818,0.641813,171,-35.183173
3,West,118,17048.6610,-16014.6922,0.669492,118,-93.935191


In [34]:
discount_profitability = (
    df.groupby("discount_band")
      .agg(
          transactions=("row_id", "count"),
          revenue=("revenue", "sum"),
          profit=("profit", "sum"),
          avg_discount=("discount", "mean")
      )
      .reset_index()
)

discount_profitability["profit_margin"] = (
    discount_profitability["profit"]
    / discount_profitability["revenue"]
    * 100
)

discount_profitability

,discount_band,transactions,revenue,profit,avg_discount,profit_margin
0,0%,4798,1.087908e+06,320987.6032,0.000000,29.505019
1,1-10%,94,5.436935e+04,9029.1770,0.100000,16.607108
2,11-20%,3709,7.921529e+05,91756.2975,0.199299,11.583155
3,21-30%,227,1.032267e+05,-10369.2774,0.300000,-10.045155
4,31-50%,310,1.953148e+05,-48447.7273,0.416097,-24.804950
5,51%+,856,6.422874e+04,-76559.0513,0.718925,-119.197502


Step 3.3 — Product Risk

In [35]:
product_risk = (
    df.groupby(
        ["product_id", "product_name", "category", "sub_category"],
        as_index=False
    )
    .agg(
        transactions=("row_id", "count"),
        revenue=("revenue", "sum"),
        profit=("profit", "sum"),
        units=("quantity", "sum"),
        avg_discount=("discount", "mean")
    )
)

In [37]:
product_risk["profit_margin"] = (
    product_risk["profit"]
    / product_risk["revenue"]
    * 100
)

product_risk["revenue_share"] = (
    product_risk["revenue"]
    / product_risk["revenue"].sum()
    * 100
)

product_risk.shape

(1894, 11)

In [38]:
loss_products = product_risk[
    product_risk["profit"] < 0
].copy()

print("Total products:", len(product_risk))
print("Loss-making products:", len(loss_products))
print(
    "Loss-making product rate:",
    len(loss_products) / len(product_risk) * 100,
    "%"
)

Total products: 1894
Loss-making products: 306
Loss-making product rate: 16.156282998944032 %


In [39]:
loss_products = loss_products.sort_values(
    "profit"
)

loss_products[
    [
        "product_id",
        "product_name",
        "category",
        "sub_category",
        "transactions",
        "revenue",
        "profit",
        "avg_discount",
        "profit_margin"
    ]
].head(10)

,product_id,product_name,category,sub_category,transactions,revenue,profit,avg_discount,profit_margin
1645,TEC-MA-10000418,Cubify CubeX 3D Printer Double Head Print,Technology,Machines,3,11099.963,-8879.9704,0.533333,-80.000000
1650,TEC-MA-10000822,Lexmark MX611dhe Monochrome Laser Printer,Technology,Machines,4,16829.901,-4589.9730,0.400000,-27.272727
1696,TEC-MA-10004125,Cubify CubeX 3D Printer Triple Head Print,Technology,Machines,1,7999.980,-3839.9904,0.500000,-48.000000
326,FUR-TA-10000198,Chromcraft Bull-Nose Wood Oval Conference Tabl...,Furniture,Tables,5,9917.640,-2876.1156,0.280000,-29.000000
344,FUR-TA-10001889,Bush Advantage Collection Racetrack Conference...,Furniture,Tables,7,9544.725,-1934.3976,0.350000,-20.266667
858,OFF-BI-10004995,GBC DocuBind P400 Electric Binding System,Office Supplies,Binders,6,17965.068,-1878.1662,0.450000,-10.454545
1669,TEC-MA-10002412,Cisco TelePresence System EX90 Videoconferenci...,Technology,Machines,1,22638.480,-1811.0784,0.500000,-8.000000
1465,OFF-SU-10002881,Martin Yale Chadless Opener Electric Letter Op...,Office Supplies,Supplies,6,16656.200,-1299.1836,0.100000,-7.800000
346,FUR-TA-10001950,Balt Solid Wood Round Tables,Furniture,Tables,4,6518.754,-1201.0581,0.200000,-18.424658
375,FUR-TA-10004289,BoxOffice By Design Rectangular and Half-Moon ...,Furniture,Tables,3,1706.250,-1148.4375,0.483333,-67.307692


In [40]:
total_product_loss = abs(
    loss_products["profit"].sum()
)

print(
    "Total product losses:",
    total_product_loss
)

loss_products["loss_amount"] = (
    loss_products["profit"].abs()
)

loss_products = loss_products.sort_values(
    "loss_amount",
    ascending=False
)

loss_products["cumulative_loss"] = (
    loss_products["loss_amount"].cumsum()
)

loss_products["cumulative_loss"] = (
    loss_products["loss_amount"].cumsum()
)

loss_products["cumulative_loss_pct"] = (
    loss_products["cumulative_loss"]
    / total_product_loss
    * 100
)

Total product losses: 77092.01719999999


In [41]:
loss_products.head(20)

,product_id,product_name,category,sub_category,transactions,revenue,profit,units,avg_discount,profit_margin,revenue_share,loss_amount,cumulative_loss,cumulative_loss_pct
1645,TEC-MA-10000418,Cubify CubeX 3D Printer Double Head Print,Technology,Machines,3,11099.9630,-8879.9704,9,0.533333,-80.000000,0.483195,8879.9704,8879.9704,11.518664
1650,TEC-MA-10000822,Lexmark MX611dhe Monochrome Laser Printer,Technology,Machines,4,16829.9010,-4589.9730,18,0.400000,-27.272727,0.732626,4589.9730,13469.9434,17.472553
1696,TEC-MA-10004125,Cubify CubeX 3D Printer Triple Head Print,Technology,Machines,1,7999.9800,-3839.9904,4,0.500000,-48.000000,0.348249,3839.9904,17309.9338,22.453601
326,FUR-TA-10000198,Chromcraft Bull-Nose Wood Oval Conference Tabl...,Furniture,Tables,5,9917.6400,-2876.1156,27,0.280000,-29.000000,0.431727,2876.1156,20186.0494,26.184358
344,FUR-TA-10001889,Bush Advantage Collection Racetrack Conference...,Furniture,Tables,7,9544.7250,-1934.3976,33,0.350000,-20.266667,0.415494,1934.3976,22120.4470,28.693564
858,OFF-BI-10004995,GBC DocuBind P400 Electric Binding System,Office Supplies,Binders,6,17965.0680,-1878.1662,27,0.450000,-10.454545,0.782042,1878.1662,23998.6132,31.129829
1669,TEC-MA-10002412,Cisco TelePresence System EX90 Videoconferenci...,Technology,Machines,1,22638.4800,-1811.0784,6,0.500000,-8.000000,0.985481,1811.0784,25809.6916,33.479072
1465,OFF-SU-10002881,Martin Yale Chadless Opener Electric Letter Op...,Office Supplies,Supplies,6,16656.2000,-1299.1836,22,0.100000,-7.800000,0.725065,1299.1836,27108.8752,35.164309
346,FUR-TA-10001950,Balt Solid Wood Round Tables,Furniture,Tables,4,6518.7540,-1201.0581,19,0.200000,-18.424658,0.283769,1201.0581,28309.9333,36.722263
375,FUR-TA-10004289,BoxOffice By Design Rectangular and Half-Moon ...,Furniture,Tables,3,1706.2500,-1148.4375,15,0.483333,-67.307692,0.074275,1148.4375,29458.3708,38.211960


In [42]:
loss_50 = (
    loss_products["cumulative_loss_pct"] <= 50
).sum() + 1

loss_80 = (
    loss_products["cumulative_loss_pct"] <= 80
).sum() + 1

loss_90 = (
    loss_products["cumulative_loss_pct"] <= 90
).sum() + 1

print("Products responsible for ~50% of losses:", loss_50)
print("Products responsible for ~80% of losses:", loss_80)
print("Products responsible for ~90% of losses:", loss_90)

Products responsible for ~50% of losses: 20
Products responsible for ~80% of losses: 69
Products responsible for ~90% of losses: 106


In [43]:
high_discount_loss_products = product_risk[
    (product_risk["avg_discount"] > 0.20) &
    (product_risk["profit"] < 0)
].copy()

print(
    "High-discount + loss-making products:",
    len(high_discount_loss_products)
)

high_discount_loss_products = (
    high_discount_loss_products
    .sort_values("profit")
)

High-discount + loss-making products: 186


In [44]:
high_discount_loss_products[
    [
        "product_id",
        "product_name",
        "category",
        "sub_category",
        "transactions",
        "revenue",
        "profit",
        "avg_discount",
        "profit_margin"
    ]
].head(15)

,product_id,product_name,category,sub_category,transactions,revenue,profit,avg_discount,profit_margin
1645,TEC-MA-10000418,Cubify CubeX 3D Printer Double Head Print,Technology,Machines,3,11099.963,-8879.9704,0.533333,-80.000000
1650,TEC-MA-10000822,Lexmark MX611dhe Monochrome Laser Printer,Technology,Machines,4,16829.901,-4589.9730,0.400000,-27.272727
1696,TEC-MA-10004125,Cubify CubeX 3D Printer Triple Head Print,Technology,Machines,1,7999.980,-3839.9904,0.500000,-48.000000
326,FUR-TA-10000198,Chromcraft Bull-Nose Wood Oval Conference Tabl...,Furniture,Tables,5,9917.640,-2876.1156,0.280000,-29.000000
344,FUR-TA-10001889,Bush Advantage Collection Racetrack Conference...,Furniture,Tables,7,9544.725,-1934.3976,0.350000,-20.266667
858,OFF-BI-10004995,GBC DocuBind P400 Electric Binding System,Office Supplies,Binders,6,17965.068,-1878.1662,0.450000,-10.454545
1669,TEC-MA-10002412,Cisco TelePresence System EX90 Videoconferenci...,Technology,Machines,1,22638.480,-1811.0784,0.500000,-8.000000
375,FUR-TA-10004289,BoxOffice By Design Rectangular and Half-Moon ...,Furniture,Tables,3,1706.250,-1148.4375,0.483333,-67.307692
372,FUR-TA-10004154,"Riverside Furniture Oval Coffee Table, Oval En...",Furniture,Tables,5,4446.175,-1147.4000,0.300000,-25.806452
1668,TEC-MA-10002210,Epson TM-T88V Direct Thermal Printer - Monochr...,Technology,Machines,2,1212.705,-1057.2300,0.550000,-87.179487


In [45]:
def classify_product_risk(row):

    loss = row["profit"] < 0
    high_discount = row["avg_discount"] > 0.20

    if loss and high_discount:
        return "High Risk"

    elif loss or high_discount:
        return "Moderate Risk"

    else:
        return "Low Risk"


product_risk["risk_level"] = (
    product_risk.apply(
        classify_product_risk,
        axis=1
    )
)

In [46]:
product_risk["risk_level"].value_counts()

risk_level
Low Risk         1338
Moderate Risk     370
High Risk         186
Name: count, dtype: int64

In [47]:
product_risk_summary = (
    product_risk
    .groupby("risk_level")
    .agg(
        products=("product_id", "count"),
        revenue=("revenue", "sum"),
        profit=("profit", "sum"),
        avg_discount=("avg_discount", "mean")
    )
    .reset_index()
)

In [48]:
product_risk_summary["profit_margin"] = (
    product_risk_summary["profit"]
    / product_risk_summary["revenue"]
    * 100
)

In [49]:
product_risk_summary

,risk_level,products,revenue,profit,avg_discount,profit_margin
0,High Risk,186,3.455396e+05,-62439.1791,0.398296,-18.070051
1,Low Risk,1338,1.416584e+06,314544.6302,0.088652,22.204446
2,Moderate Risk,370,5.350772e+05,34291.5706,0.248707,6.408714


In [51]:
product_category_risk = (
    product_risk
    .groupby(["category", "risk_level"])
    .agg(
        products=("product_id", "count"),
        revenue=("revenue", "sum"),
        profit=("profit", "sum"),
        avg_discount=("avg_discount", "mean")
    )
    .reset_index()
)

product_category_risk["profit_margin"] = (
    product_category_risk["profit"]
    / product_category_risk["revenue"]
    * 100
)

In [52]:
product_category_risk

,category,risk_level,products,revenue,profit,avg_discount,profit_margin
0,Furniture,High Risk,85,211933.2161,-29940.1153,0.297304,-14.127146
1,Furniture,Low Risk,223,386355.0882,51840.4000,0.113043,13.417812
2,Furniture,Moderate Risk,75,143711.4910,-3449.0119,0.216526,-2.399956
3,Office Supplies,High Risk,66,41784.7280,-6131.0936,0.493745,-14.673049
4,Office Supplies,Low Risk,808,395143.8090,102475.5708,0.076638,25.933741
5,Office Supplies,Moderate Risk,224,282118.4950,26146.3236,0.267882,9.267852
6,Technology,High Risk,35,91821.6350,-26367.9702,0.463571,-28.716511
7,Technology,Low Risk,307,635085.1360,160228.6594,0.102555,25.229477
8,Technology,Moderate Risk,71,109247.2620,11594.2589,0.222203,10.612860


Step 3.4 — Regional Risk Analysis

In [53]:
region_risk = (
    df.groupby("region", as_index=False)
    .agg(
        transactions=("row_id", "count"),
        revenue=("revenue", "sum"),
        profit=("profit", "sum"),
        units=("quantity", "sum"),
        avg_discount=("discount", "mean")
    )
)

In [54]:
region_risk["profit_margin"] = (
    region_risk["profit"]
    / region_risk["revenue"]
    * 100
)

In [55]:
loss_counts = (
    df[df["profit"] < 0]
    .groupby("region")
    .size()
    .reset_index(name="loss_transactions")
)

region_risk = region_risk.merge(
    loss_counts,
    on="region",
    how="left"
)

region_risk["loss_transactions"] = (
    region_risk["loss_transactions"]
    .fillna(0)
    .astype(int)
)

In [56]:
region_risk["loss_transaction_rate"] = (
    region_risk["loss_transactions"]
    / region_risk["transactions"]
    * 100
)

In [57]:
region_risk

,region,transactions,revenue,profit,units,avg_discount,profit_margin,loss_transactions,loss_transaction_rate
0,Central,2323,501239.8908,39706.3625,8780,0.240353,7.921629,741,31.898407
1,East,2848,678781.2400,91522.7800,10618,0.145365,13.483399,553,19.417135
2,South,1620,391721.9050,46749.4303,6209,0.147253,11.934342,259,15.987654
3,West,3203,725457.8245,108418.4489,12266,0.109335,14.944831,318,9.928192


In [58]:
region_risk["revenue_share"] = (
    region_risk["revenue"]
    / region_risk["revenue"].sum()
    * 100
)

region_risk["profit_share"] = (
    region_risk["profit"]
    / region_risk["profit"].sum()
    * 100
)

region_risk

,region,transactions,revenue,profit,units,avg_discount,profit_margin,loss_transactions,loss_transaction_rate,revenue_share,profit_share
0,Central,2323,501239.8908,39706.3625,8780,0.240353,7.921629,741,31.898407,21.819594,13.864098
1,East,2848,678781.2400,91522.7800,10618,0.145365,13.483399,553,19.417135,29.548188,31.956610
2,South,1620,391721.9050,46749.4303,6209,0.147253,11.934342,259,15.987654,17.052140,16.323295
3,West,3203,725457.8245,108418.4489,12266,0.109335,14.944831,318,9.928192,31.580078,37.855997


In [59]:
high_discount_region = (
    df[df["discount"] > 0.20]
    .groupby("region", as_index=False)
    .agg(
        high_discount_transactions=("row_id", "count"),
        revenue=("revenue", "sum"),
        profit=("profit", "sum"),
        avg_discount=("discount", "mean")
    )
)

high_discount_region = high_discount_region.merge(
    region_risk[
        ["region", "transactions"]
    ],
    on="region",
    how="left"
)

high_discount_region["high_discount_rate"] = (
    high_discount_region["high_discount_transactions"]
    / high_discount_region["transactions"]
    * 100
)

high_discount_region

,region,high_discount_transactions,revenue,profit,avg_discount,transactions,high_discount_rate
0,Central,643,129134.0988,-52392.2744,0.606128,2323,27.679724
1,East,461,145870.3970,-42088.6076,0.507592,2848,16.186798
2,South,171,70716.9930,-24880.4818,0.641813,1620,10.555556
3,West,118,17048.6610,-16014.6922,0.669492,3203,3.684046


In [60]:
regional_losses = (
    df[df["profit"] < 0]
    .groupby("region", as_index=False)
    .agg(
        loss_transactions=("row_id", "count"),
        loss_revenue=("revenue", "sum"),
        total_loss=("profit", "sum"),
        avg_discount=("discount", "mean")
    )
)

In [61]:
regional_losses["loss_amount"] = (
    regional_losses["total_loss"].abs()
)

regional_losses

,region,loss_transactions,loss_revenue,total_loss,avg_discount,loss_amount
0,Central,741,141282.6588,-56314.8850,0.549447,56314.8850
1,East,553,160864.0120,-49590.6075,0.446835,49590.6075
2,South,259,91635.1850,-27504.8323,0.491699,27504.8323
3,West,318,74925.2990,-22720.9609,0.371541,22720.9609


In [62]:
region_category_risk = (
    df.groupby(
        ["region", "category"],
        as_index=False
    )
    .agg(
        transactions=("row_id", "count"),
        revenue=("revenue", "sum"),
        profit=("profit", "sum"),
        avg_discount=("discount", "mean")
    )
)

region_category_risk["profit_margin"] = (
    region_category_risk["profit"]
    / region_category_risk["revenue"]
    * 100
)

In [63]:
region_category_risk[
    region_category_risk["profit"] < 0
].sort_values("profit")

,region,category,transactions,revenue,profit,avg_discount,profit_margin
0,Central,Furniture,481,163797.1638,-2871.0494,0.29738,-1.752808


In [64]:
df["high_discount_flag"] = (
    df["discount"] > 0.20
)

region_discount_risk = (
    df.groupby("region")
    .agg(
        transactions=("row_id", "count"),
        revenue=("revenue", "sum"),
        profit=("profit", "sum"),
        high_discount_transactions=(
            "high_discount_flag",
            "sum"
        )
    )
    .reset_index()
)

region_discount_risk["high_discount_rate"] = (
    region_discount_risk["high_discount_transactions"]
    / region_discount_risk["transactions"]
    * 100
)

In [65]:
worst_region_category = (
    region_category_risk
    .sort_values("profit_margin")
)

In [66]:
worst_region_category[
    [
        "region",
        "category",
        "transactions",
        "revenue",
        "profit",
        "avg_discount",
        "profit_margin"
    ]
].head(10)

,region,category,transactions,revenue,profit,avg_discount,profit_margin
0,Central,Furniture,481,163797.1638,-2871.0494,0.297380,-1.752808
3,East,Furniture,601,208291.2040,3046.1658,0.154077,1.462455
9,West,Furniture,707,252612.7435,11504.9503,0.131400,4.554382
1,Central,Office Supplies,1422,167026.4150,8879.9799,0.252743,5.316512
6,South,Furniture,332,117298.6840,6771.2061,0.121536,5.772619
8,South,Technology,293,148771.9080,19991.8314,0.107850,13.437908
7,South,Office Supplies,995,125651.3130,19986.3928,0.167437,15.906235
11,West,Technology,599,251991.8320,44303.6496,0.133890,17.581383
5,East,Technology,535,264973.9810,47462.0351,0.143364,17.911961
2,Central,Technology,420,170416.3120,33697.4320,0.133095,19.773595


In [67]:
def classify_region_risk(row):

    if row["profit_margin"] < 0:
        return "High Risk"

    elif row["profit_margin"] < 10:
        return "Moderate Risk"

    else:
        return "Low Risk"


region_risk["risk_level"] = (
    region_risk.apply(
        classify_region_risk,
        axis=1
    )
)

In [68]:
region_risk

,region,transactions,revenue,profit,units,avg_discount,profit_margin,loss_transactions,loss_transaction_rate,revenue_share,profit_share,risk_level
0,Central,2323,501239.8908,39706.3625,8780,0.240353,7.921629,741,31.898407,21.819594,13.864098,Moderate Risk
1,East,2848,678781.2400,91522.7800,10618,0.145365,13.483399,553,19.417135,29.548188,31.956610,Low Risk
2,South,1620,391721.9050,46749.4303,6209,0.147253,11.934342,259,15.987654,17.052140,16.323295,Low Risk
3,West,3203,725457.8245,108418.4489,12266,0.109335,14.944831,318,9.928192,31.580078,37.855997,Low Risk


In [69]:
region_risk_final = region_risk[
    [
        "region",
        "transactions",
        "revenue",
        "profit",
        "units",
        "avg_discount",
        "profit_margin",
        "loss_transactions",
        "loss_transaction_rate",
        "revenue_share",
        "profit_share",
        "risk_level"
    ]
].sort_values(
    "profit_margin"
)

In [70]:
region_risk_final

,region,transactions,revenue,profit,units,avg_discount,profit_margin,loss_transactions,loss_transaction_rate,revenue_share,profit_share,risk_level
0,Central,2323,501239.8908,39706.3625,8780,0.240353,7.921629,741,31.898407,21.819594,13.864098,Moderate Risk
2,South,1620,391721.9050,46749.4303,6209,0.147253,11.934342,259,15.987654,17.052140,16.323295,Low Risk
1,East,2848,678781.2400,91522.7800,10618,0.145365,13.483399,553,19.417135,29.548188,31.956610,Low Risk
3,West,3203,725457.8245,108418.4489,12266,0.109335,14.944831,318,9.928192,31.580078,37.855997,Low Risk


Step 3.4 — Final Regional Diagnosis

In [71]:
region_category_risk.sort_values(
    ["region", "profit_margin"]
)

,region,category,transactions,revenue,profit,avg_discount,profit_margin
0,Central,Furniture,481,163797.1638,-2871.0494,0.297380,-1.752808
1,Central,Office Supplies,1422,167026.4150,8879.9799,0.252743,5.316512
2,Central,Technology,420,170416.3120,33697.4320,0.133095,19.773595
3,East,Furniture,601,208291.2040,3046.1658,0.154077,1.462455
5,East,Technology,535,264973.9810,47462.0351,0.143364,17.911961
4,East,Office Supplies,1712,205516.0550,41014.5791,0.142932,19.956873
6,South,Furniture,332,117298.6840,6771.2061,0.121536,5.772619
8,South,Technology,293,148771.9080,19991.8314,0.107850,13.437908
7,South,Office Supplies,995,125651.3130,19986.3928,0.167437,15.906235
9,West,Furniture,707,252612.7435,11504.9503,0.131400,4.554382


In [72]:
central_category_risk = (
    region_category_risk[
        region_category_risk["region"] == "Central"
    ]
    .sort_values("profit_margin")
)

central_category_risk

,region,category,transactions,revenue,profit,avg_discount,profit_margin
0,Central,Furniture,481,163797.1638,-2871.0494,0.297380,-1.752808
1,Central,Office Supplies,1422,167026.4150,8879.9799,0.252743,5.316512
2,Central,Technology,420,170416.3120,33697.4320,0.133095,19.773595


In [73]:
central_losses = (
    df[
        (df["region"] == "Central") &
        (df["profit"] < 0)
    ]
    .groupby("category", as_index=False)
    .agg(
        loss_transactions=("row_id", "count"),
        revenue=("revenue", "sum"),
        profit=("profit", "sum"),
        avg_discount=("discount", "mean")
    )
)

central_losses["loss_amount"] = (
    central_losses["profit"].abs()
)

central_losses["loss_share"] = (
    central_losses["loss_amount"]
    / central_losses["loss_amount"].sum()
    * 100
)

central_losses.sort_values(
    "loss_amount",
    ascending=False
)

,category,loss_transactions,revenue,profit,avg_discount,loss_amount,loss_share
1,Office Supplies,375,33072.5080,-33484.1655,0.680000,33484.1655,59.458819
0,Furniture,317,85278.5008,-19554.3653,0.443659,19554.3653,34.723262
2,Technology,49,22931.6500,-3276.3542,0.234694,3276.3542,5.817919


Step 3.5 — Build the Discount × Profitability Analysis

In [74]:
df["discount_group"] = pd.cut(
    df["discount"],
    bins=[-0.001, 0, 0.1, 0.2, 0.3, 0.5, 1.0],
    labels=[
        "0%",
        "1-10%",
        "11-20%",
        "21-30%",
        "31-50%",
        "51%+"
    ]
)

In [75]:
discount_profit_analysis = (
    df.groupby("discount_group", observed=True)
    .agg(
        transactions=("row_id", "count"),
        revenue=("revenue", "sum"),
        profit=("profit", "sum"),
        avg_profit=("profit", "mean"),
        loss_transactions=("profit", lambda x: (x < 0).sum())
    )
    .reset_index()
)

In [77]:
discount_profit_analysis["profit_margin"] = (
    discount_profit_analysis["profit"]
    / discount_profit_analysis["revenue"]
    * 100
)

discount_profit_analysis["loss_rate"] = (
    discount_profit_analysis["loss_transactions"]
    / discount_profit_analysis["transactions"]
    * 100
)

discount_profit_analysis

,discount_group,transactions,revenue,profit,avg_profit,loss_transactions,profit_margin,loss_rate
0,0%,4798,1.087908e+06,320987.6032,66.900292,0,29.505019,0.000000
1,1-10%,94,5.436935e+04,9029.1770,96.055074,4,16.607108,4.255319
2,11-20%,3709,7.921529e+05,91756.2975,24.738824,519,11.583155,13.992990
3,21-30%,227,1.032267e+05,-10369.2774,-45.679636,208,-10.045155,91.629956
4,31-50%,310,1.953148e+05,-48447.7273,-156.282991,284,-24.804950,91.612903
5,51%+,856,6.422874e+04,-76559.0513,-89.438144,856,-119.197502,100.000000


Step 3.6 — Final EDA: Correlation Analysis

In [79]:
# The US Superstore data has no shipping cost column,
# so use shipping time (days from order to shipment) instead.
# "revenue" is left out because it is identical to "sales".
df["shipping_days"] = (df["ship_date"] - df["order_date"]).dt.days

numeric_cols = [
    "sales",
    "quantity",
    "discount",
    "shipping_days",
    "profit"
]

correlation = (
    df[numeric_cols]
    .corr()["profit"]
    .sort_values(ascending=False)
)

correlation

profit           1.000000
sales            0.479064
quantity         0.066253
shipping_days   -0.004649
discount        -0.219487
Name: profit, dtype: float64